<a href="https://colab.research.google.com/github/RIVALX-1/sae_dlm_cross_entropy/blob/main/sae_dlm_cross_entropy_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!git clone https://github.com/RIVALX-1/sae_dlm_cross_entropy
%cd sae_dlm_cross_entropy

Cloning into 'sae_dlm_cross_entropy'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), done.
/content/sae_dlm_cross_entropy


In [3]:
# ⚠️ Fill in YOUR details before running
!git config --global user.email "sairaghuram@gmail.com"
!git config --global user.name "Sai Raghuram Jandhyam"

In [ ]:
!pip install transformers==4.46.2
!pip install torch
!pip install git+https://github.com/saprmarks/dictionary_learning.git
!pip install huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 121.7 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.5.0
    Uninstalling huggingface_hub-1.5.0:
      Successfully uninstalled huggingface_hub-1.5.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
  Cloning https://github.com/saprmarks/dictionary_learning.git to /tmp/pip-req-build-4f4oz7ux
  Running command git clone --filter=blob:none --quie

In [ ]:
from dictionary_learning import AutoEncoder, utils
from transformers import AutoModel, AutoTokenizer
from huggingface_hub import hf_hub_download
import torch
from huggingface_hub import login
login()

In [ ]:
from dictionary_learning.trainers.top_k import AutoEncoderTopK

class TopKSae(AutoEncoderTopK):

    @classmethod
    def from_ae_pt(cls, repo_id: str, layer: int) -> "TopKSae":
        path = hf_hub_download(
            repo_id=repo_id,
            filename=f"saes_mask_Dream-org_Dream-v0-Base-7B_top_k/"
                     f"resid_post_layer_{layer}/trainer_0/ae.pt"
        )
        ckpt = torch.load(path, map_location="cpu")

        dict_size, activation_dim = ckpt["encoder.weight"].shape
        k = int(ckpt["k"])

        sae = cls(activation_dim, dict_size, k)

        # let AutoEncoderTopK load its own keys as-is
        sae.load_state_dict(ckpt, strict=False)
        return sae

sae = TopKSae.from_ae_pt("AwesomeInterpretability/dlm-mask-topk-sae", layer=5)
print(sae)
print(sae.b_dec)

TopKSae(
  (decoder): Linear(in_features=16384, out_features=3584, bias=False)
  (encoder): Linear(in_features=3584, out_features=16384, bias=True)
)
Parameter containing:
tensor([-0.0030,  0.0445, -0.0761,  ..., -0.0028, -0.0177,  0.0708],
       requires_grad=True)


In [ ]:
print('We have access to a GPU:', torch.cuda.is_available())
!nvidia-smi

We have access to a GPU: True
Mon Mar  9 22:15:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             32W /   70W |   14867MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------

In [ ]:
from huggingface_hub import list_repo_files

files = list_repo_files("AwesomeInterpretability/dlm-mask-topk-sae")

for f in files:
    print(f)

.gitattributes
README.md
saes_mask_Dream-org_Dream-v0-Base-7B_top_k/resid_post_layer_1/trainer_0/ae.pt
saes_mask_Dream-org_Dream-v0-Base-7B_top_k/resid_post_layer_1/trainer_0/config.json
saes_mask_Dream-org_Dream-v0-Base-7B_top_k/resid_post_layer_1/trainer_0/eval_results.json
saes_mask_Dream-org_Dream-v0-Base-7B_top_k/resid_post_layer_1/trainer_1/ae.pt
saes_mask_Dream-org_Dream-v0-Base-7B_top_k/resid_post_layer_1/trainer_1/config.json
saes_mask_Dream-org_Dream-v0-Base-7B_top_k/resid_post_layer_1/trainer_1/eval_results.json
saes_mask_Dream-org_Dream-v0-Base-7B_top_k/resid_post_layer_1/trainer_2/ae.pt
saes_mask_Dream-org_Dream-v0-Base-7B_top_k/resid_post_layer_1/trainer_2/config.json
saes_mask_Dream-org_Dream-v0-Base-7B_top_k/resid_post_layer_1/trainer_2/eval_results.json
saes_mask_Dream-org_Dream-v0-Base-7B_top_k/resid_post_layer_1/trainer_3/ae.pt
saes_mask_Dream-org_Dream-v0-Base-7B_top_k/resid_post_layer_1/trainer_3/config.json
saes_mask_Dream-org_Dream-v0-Base-7B_top_k/resid_post_lay

In [ ]:
layer = 5

ae_path = hf_hub_download(
    repo_id="AwesomeInterpretability/dlm-mask-topk-sae",
    filename=f"saes_mask_Dream-org_Dream-v0-Base-7B_top_k/resid_post_layer_{layer}/trainer_0/ae.pt"
)

saes_mask_Dream-org_Dream-v0-Base-7B_top(…):   0%|          | 0.00/470M [00:00<?, ?B/s]

In [ ]:
#downloading all the layer files at once

#files = list_repo_files("AwesomeInterpretability/dlm-mask-topk-sae")

#ae_files = [f for f in files if f.endswith("ae.pt")]

#for f in ae_files:
#    path = hf_hub_download(
#        repo_id="AwesomeInterpretability/dlm-mask-topk-sae",
#        filename=f
#    )

In [ ]:
ae = torch.load(ae_path)
dict_size, activation_dim = ae["encoder.weight"].shape

autoencoder = AutoEncoder(activation_dim, dict_size)

ae['bias'] = ae['b_dec']
del ae['b_dec']
del ae['k']
del ae['threshold']

autoencoder.load_state_dict(ae)

<All keys matched successfully>

In [ ]:
model = AutoModel.from_pretrained(
    "Dream-org/Dream-v0-Base-7B",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)

config.json:   0%|          | 0.00/874 [00:00<?, ?B/s]

configuration_dream.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/Dream-org/Dream-v0-Base-7B:
- configuration_dream.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_dream.py: 0.00B [00:00, ?B/s]

generation_utils.py: 0.00B [00:00, ?B/s]

...

model-00002-of-00004.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.33G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
model = model.cuda()
model.eval()

DreamModel(
  (model): DreamBaseModel(
    (embed_tokens): Embedding(152064, 3584, padding_idx=151643)
    (layers): ModuleList(
      (0-27): 28 x DreamDecoderLayer(
        (self_attn): DreamSdpaAttention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
          (rotary_emb): DreamRotaryEmbedding()
        )
        (mlp): DreamMLP(
          (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): DreamRMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): DreamRMSNorm((3584,), eps=1e-06)
      )

In [ ]:
def splice_sae(module, input, output):
    with torch.no_grad():
        encoded = autoencoder.encode(output)
        reconstructed = autoencoder.decode(encoded)
    return reconstructed


In [ ]:
model_path = "GSAI-ML/LLaDA-8B-Base"
messages = [
    {"role": "user", "content": "Please write a Python class that implements a PyTorch trainer capable of training a model on a toy dataset."}
]
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
inputs = tokenizer.apply_chat_template(
    messages, return_tensors="pt", return_dict=True, add_generation_prompt=True
)
input_ids = inputs.input_ids.to(device="cuda")
attention_mask = inputs.attention_mask.to(device="cuda")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/766 [00:00<?, ?B/s]

In [ ]:
from transformers import AutoModel, AutoTokenizer
import torch

model_path = "GSAI-ML/LLaDA-8B-Base"

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    trust_remote_code=True
)

model = AutoModel.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    device_map="auto"
)

model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/766 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

model-00005-of-00006.safetensors:   0%|          | 0.00/2.92G [00:00<?, ?B/s]

model-00006-of-00006.safetensors:   0%|          | 0.00/2.11G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

LLaDAModelLM(
  (model): LLaDAModel(
    (transformer): ModuleDict(
      (wte): Embedding(126464, 4096)
      (emb_drop): Dropout(p=0.0, inplace=False)
      (ln_f): RMSLayerNorm()
      (blocks): ModuleList(
        (0-31): 32 x LLaDALlamaBlock(
          (dropout): Dropout(p=0.0, inplace=False)
          (act): SiLU()
          (attn_out): Linear(in_features=4096, out_features=4096, bias=False)
          (ff_out): Linear(in_features=12288, out_features=4096, bias=False)
          (rotary_emb): RotaryEmbedding()
          (attn_norm): RMSLayerNorm()
          (ff_norm): RMSLayerNorm()
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (ff_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (up_proj): Linear(in_features=4096, out_features=12288, bias=False)
        )
      )
    

In [ ]:
prompt = "The capital of France is"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask
    )

print(output.logits.shape)

torch.Size([1, 5, 126464])


In [ ]:
import torch
import torch.nn as nn

In [ ]:
def geometric_pmf(p: float, k: torch.Tensor) -> torch.Tensor:
    return p * torch.pow(1.0 - p, k)

def sample_exact_k_mask(maskable_positions: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    B, L = maskable_positions.shape
    device = maskable_positions.device
    mask = torch.zeros_like(maskable_positions, dtype=torch.bool)

    for b in range(B):
        valid_idx = torch.where(maskable_positions[b])[0]
        m = valid_idx.numel()
        if m == 0:
            continue

        k = int(torch.round(t[b] * m).item())
        k = max(1, min(k, m))

        perm = torch.randperm(m, device=device)
        chosen = valid_idx[perm[:k]]
        mask[b, chosen] = True

    return mask

In [ ]:
def compute_dlm_cross_entropy(
    model,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    mask_token_id: int,
    loss_mask: torch.Tensor | None = None,
    mode: str = "llada",
    exact_k_masking: bool = True,
    cart_p: float = 0.5,
    position_ids: torch.Tensor | None = None,
):
    device = input_ids.device
    B, L = input_ids.shape

    if loss_mask is None:
        maskable_positions = attention_mask.bool()
    else:
        maskable_positions = attention_mask.bool() & loss_mask.bool()

    t = torch.rand(B, device=device).clamp_min(1e-4)

    if exact_k_masking:
        mask_positions = sample_exact_k_mask(maskable_positions, t)
    else:
        rand = torch.rand(B, L, device=device)
        mask_positions = (rand < t[:, None]) & maskable_positions
        for b in range(B):
            if maskable_positions[b].any() and not mask_positions[b].any():
                first_valid = torch.where(maskable_positions[b])[0][0]
                mask_positions[b, first_valid] = True

    corrupted_input_ids = input_ids.clone()
    corrupted_input_ids[mask_positions] = mask_token_id

    forward_kwargs = {
        "input_ids": corrupted_input_ids,
        "attention_mask": attention_mask,
    }
    if position_ids is not None:
        forward_kwargs["position_ids"] = position_ids

    outputs = model(**forward_kwargs)
    logits = outputs.logits

    ce_fct = nn.CrossEntropyLoss(reduction="none")
    per_token_ce = ce_fct(
        logits.view(-1, logits.size(-1)),
        input_ids.view(-1)
    ).view(B, L)

    weight_map = mask_positions.float()

    if mode in ("llada", "dream_base"):
        weight_map = weight_map * (1.0 / t)[:, None]

    elif mode == "dream_cart":
        visible_positions = (~mask_positions) & maskable_positions
        idx = torch.arange(L, device=device)
        cart_weights = torch.zeros(B, L, device=device)

        for b in range(B):
            visible_idx = torch.where(visible_positions[b])[0]
            if visible_idx.numel() == 0:
                cart_weights[b] = 1.0
                continue

            n_idx = idx[:, None]
            i_idx = visible_idx[None, :]
            k = (n_idx - i_idx).abs() - 1
            k = torch.clamp(k, min=0)

            geo_vals = geometric_pmf(cart_p, k.float())
            cart_weights[b] = 0.5 * geo_vals.sum(dim=1)

        weight_map = weight_map * (1.0 / t)[:, None] * cart_weights

    else:
        raise ValueError("mode must be one of: 'llada', 'dream_base', 'dream_cart'")

    weighted_loss_sum = (per_token_ce * weight_map).sum()
    normalizer = weight_map.sum().clamp_min(1e-8)
    loss = weighted_loss_sum / normalizer

    return {
        "loss": loss,
        "t": t,
        "corrupted_input_ids": corrupted_input_ids,
        "mask_positions": mask_positions,
        "logits": logits,
        "per_token_ce": per_token_ce,
        "weight_map": weight_map,
    }

In [ ]:
prompt = "The capital of France is Paris."

inputs = tokenizer(prompt, return_tensors="pt")
input_ids = inputs.input_ids.to(model.device)
attention_mask = inputs.attention_mask.to(model.device)

dlm_mask_token_id = tokenizer.mask_token_id
if dlm_mask_token_id is None:
    if tokenizer.eos_token_id is not None:
        dlm_mask_token_id = tokenizer.eos_token_id
        print(f"Warning: tokenizer.mask_token_id is None. Using tokenizer.eos_token_id ({dlm_mask_token_id}) as mask_token_id.")
    elif tokenizer.pad_token_id is not None:
        dlm_mask_token_id = tokenizer.pad_token_id
        print(f"Warning: tokenizer.mask_token_id is None and tokenizer.eos_token_id is None. Using tokenizer.pad_token_id ({dlm_mask_token_id}) as mask_token_id.")
    else:
        # Fallback to an arbitrary token ID if no suitable special token is found.
        # This is a risky fallback and might not be semantically correct for the DLM task.
        dlm_mask_token_id = 0 # Assuming 0 is a valid token ID in the vocabulary
        print(f"Warning: No suitable mask, eos, or pad token found. Using token ID 0 as mask_token_id. This might lead to unexpected behavior.")

print("mask_token_id:", dlm_mask_token_id)

result = compute_dlm_cross_entropy(
    model=model,
    input_ids=input_ids,
    attention_mask=attention_mask,
    mask_token_id=dlm_mask_token_id,
    mode="llada",
    exact_k_masking=True
)

print("Loss:", result["loss"].item())

mask_token_id: 126081
Loss: 4.500000476837158


In [ ]:
!nvidia-smi

Tue Mar 10 01:47:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P0             28W /   70W |   12327MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from datasets import load_dataset

ds = load_dataset("common-pile/wikiteam")

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/978 [00:00<?, ?it/s]

00000_wiki.jsonl.gz:   0%|          | 0.00/124M [00:00<?, ?B/s]

00001_wiki.jsonl.gz:   0%|          | 0.00/129M [00:00<?, ?B/s]

00002_wiki.jsonl.gz:   0%|          | 0.00/121M [00:00<?, ?B/s]

...

00216_wiki.jsonl.gz:   0%|          | 0.00/2.20M [00:00<?, ?B/s]

00217_wiki.jsonl.gz:   0%|          | 0.00/2.24M [00:00<?, ?B/s]

00218_wiki.jsonl.gz:   0%|          | 0.00/1.86M [00:00<?, ?B/s]

00219_wiki.jsonl.gz:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

KeyboardInterrupt: 

In [ ]:
!pip install transformers datasets accelerate

In [ ]:
from datasets import load_dataset

ds = load_dataset(
    "common-pile/wikiteam",
    split="train",
    streaming=True
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/978 [00:00<?, ?it/s]

In [ ]:
from itertools import islice

subset = list(islice(ds, 10))

In [ ]:
text = subset[0]["text"]
print(text[:500])

1

About • Sandbox • Current Events • Create Article • Rules


In [ ]:
!pip install bitsandbytes==0.43.2
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121


In [ ]:
from datasets import load_dataset
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig
import torch

# Clear memory from any previous models
if 'model' in globals() and model is not None:
    del model
    torch.cuda.empty_cache()

model_path = "GSAI-ML/LLaDA-8B-Base"

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    trust_remote_code=True
)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

model = AutoModel.from_pretrained(
    model_path,
    quantization_config=quant_config,
    trust_remote_code=True,
    device_map="auto"
)

model.eval()

ds = load_dataset("common-pile/wikiteam", split="train", streaming=True)

text = next(iter(ds))["text"]

inputs = tokenizer(text[:100], return_tensors="pt").to(model.device)
outputs = model(**inputs)

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/978 [00:00<?, ?it/s]

torch.Size([1, 15, 126464])

True
1
{'model.transformer.wte': 'cpu', 'model.transformer.emb_drop': 'cpu', 'model.transformer.ln_f': 'cpu', 'model.transformer.blocks.0': 'cpu', 'model.transformer.blocks.1': 'cpu', 'model.transformer.blocks.2': 'cpu', 'model.transformer.blocks.3': 'cpu', 'model.transformer.blocks.4': 'cpu', 'model.transformer.blocks.5': 'cpu', 'model.transformer.blocks.6': 'cpu', 'model.transformer.blocks.7': 'cpu', 'model.transformer.blocks.8': 'cpu', 'model.transformer.blocks.9': 'cpu', 'model.transformer.blocks.10': 'cpu', 'model.transformer.blocks.11': 'cpu', 'model.transformer.blocks.12': 'cpu', 'model.transformer.blocks.13': 'cpu', 'model.transformer.blocks.14': 'cpu', 'model.transformer.blocks.15': 'cpu', 'model.transformer.blocks.16': 'cpu', 'model.transformer.blocks.17': 'disk', 'model.transformer.blocks.18': 'disk', 'model.transformer.blocks.19': 'disk', 'model.transformer.blocks.20': 'disk', 'model.transformer.blocks.21': 'disk', 'model.transformer.blocks.22': 'disk', 'model.transformer.b

/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1146: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  return isinstance(obj, torch.Tensor)


CUDA tensors: 253
<class 'torch.nn.parameter.Parameter'> torch.Size([126464, 4096]) torch.bfloat16
<class 'torch.nn.parameter.Parameter'> torch.Size([4096]) torch.bfloat16
<class 'torch.nn.parameter.Parameter'> torch.Size([4096, 4096]) torch.bfloat16
<class 'torch.nn.parameter.Parameter'> torch.Size([4096, 4096]) torch.bfloat16
<class 'torch.nn.parameter.Parameter'> torch.Size([4096, 4096]) torch.bfloat16
<class 'torch.nn.parameter.Parameter'> torch.Size([4096, 4096]) torch.bfloat16
<class 'torch.nn.parameter.Parameter'> torch.Size([4096, 4096]) torch.bfloat16
<class 'torch.nn.parameter.Parameter'> torch.Size([4096, 4096]) torch.bfloat16
<class 'torch.nn.parameter.Parameter'> torch.Size([4096, 4096]) torch.bfloat16
<class 'torch.nn.parameter.Parameter'> torch.Size([4096, 4096]) torch.bfloat16
<class 'torch.nn.parameter.Parameter'> torch.Size([4096, 4096]) torch.bfloat16
<class 'torch.nn.parameter.Parameter'> torch.Size([4096, 4096]) torch.bfloat16
<class 'torch.nn.parameter.Parameter'>